# Stage 5 — Fairness and Ethics Analysis

Subgroup performance is evaluated on the untouched, expired-patient-excluded test set at the deployed 0.12 decision threshold.

## Load model and evaluate the deployed operating point

**Bias-source and mitigation placeholder:** Record why a low threshold is appropriate clinically and how false positives will be managed.

In [1]:
from pathlib import Path

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
FINAL_THRESHOLD = 0.12

pipeline = joblib.load(RESULTS_DIR / 'xgboost.joblib')
X_test = pd.read_parquet(DATA_DIR / 'X_test.parquet')
y_test = pd.read_parquet(DATA_DIR / 'y_test.parquet')['readmitted_30']
probabilities = pipeline.predict_proba(X_test)[:, 1]
predictions = (probabilities >= FINAL_THRESHOLD).astype(int)

overall_metrics = {
    'precision': precision_score(y_test, predictions, zero_division=0),
    'recall': recall_score(y_test, predictions, zero_division=0),
    'f1': f1_score(y_test, predictions, zero_division=0),
}
print(f"Overall test metrics at threshold {FINAL_THRESHOLD:.2f}:")
for metric, value in overall_metrics.items():
    print(f"  {metric}: {value:.4f}")

Overall test metrics at threshold 0.12:
  precision: 0.1739
  recall: 0.6413
  f1: 0.2736


## Reconstruct demographic labels

**Bias-source and mitigation placeholder:** Document how source-data collection and encoding choices may affect demographic representation.

In [2]:
def one_hot_to_label(frame, prefix, fallback='Other/Unknown'):
    columns = [column for column in frame.columns if column.startswith(prefix)]
    if not columns:
        raise ValueError(f'No columns found with prefix: {prefix}')
    values = frame[columns].to_numpy()
    active = values.argmax(axis=1)
    labels = np.array([columns[index].removeprefix(prefix) for index in active], dtype=object)
    labels[values.max(axis=1) < 0.5] = fallback
    return pd.Series(labels, index=frame.index), columns

race_labels, race_columns = one_hot_to_label(X_test, 'cat__race_')
gender_labels, gender_columns = one_hot_to_label(X_test, 'cat__gender_')

age_levels = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)', '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']
age_column = next(column for column in X_test.columns if column.endswith('age_ordinal'))
preprocessing_pipeline = joblib.load(RESULTS_DIR / 'preprocessing_pipeline.joblib')
numeric_features = list(preprocessing_pipeline.named_steps['preprocessor'].transformers_[0][2])
age_position = numeric_features.index('age_ordinal')
scaler = preprocessing_pipeline.named_steps['preprocessor'].named_transformers_['num'].named_steps['scaler']
age_codes = np.rint(X_test[age_column] * scaler.scale_[age_position] + scaler.mean_[age_position]).astype(int)
age_codes = age_codes.clip(0, len(age_levels) - 1)
age_labels = pd.Series([age_levels[code] for code in age_codes], index=X_test.index)

demographics = pd.DataFrame({'race': race_labels, 'gender': gender_labels, 'age_band': age_labels})
print('Race columns:', race_columns)
print('Gender columns:', gender_columns)
print('Age feature:', age_column)

Race columns: ['cat__race_AfricanAmerican', 'cat__race_Asian', 'cat__race_Caucasian', 'cat__race_Hispanic', 'cat__race_Other', 'cat__race_Unknown']
Gender columns: ['cat__gender_Female', 'cat__gender_Male', 'cat__gender_Unknown/Invalid']
Age feature: num__age_ordinal


## Race subgroup metrics

**Bias-source and mitigation placeholder:** Discuss observed race-group differences, data limitations, and any monitoring or mitigation plan.

In [3]:
def subgroup_metrics(labels, demographic_type):
    rows = []
    for subgroup, indices in labels.groupby(labels).groups.items():
        actual = y_test.loc[indices]
        predicted = predictions[indices]
        rows.append({
            'demographic_type': demographic_type,
            'subgroup': subgroup,
            'support': len(indices),
            'positive_base_rate': actual.mean(),
            'precision': precision_score(actual, predicted, zero_division=0),
            'recall': recall_score(actual, predicted, zero_division=0),
            'f1': f1_score(actual, predicted, zero_division=0),
        })
    return pd.DataFrame(rows).sort_values('subgroup').reset_index(drop=True)

race_metrics = subgroup_metrics(demographics['race'], 'race')
print(race_metrics.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

demographic_type        subgroup  support  positive_base_rate  precision  recall     f1
            race AfricanAmerican     2785              0.1070     0.1705  0.6477 0.2699
            race           Asian      117              0.1795     0.3235  0.5238 0.4000
            race       Caucasian    11221              0.1131     0.1712  0.6493 0.2710
            race        Hispanic      282              0.1135     0.2500  0.5938 0.3519
            race           Other      242              0.1157     0.2388  0.5714 0.3368
            race         Unknown      350              0.1029     0.1889  0.4722 0.2698


## Gender subgroup metrics

**Bias-source and mitigation placeholder:** Discuss observed gender-group differences, data limitations, and any monitoring or mitigation plan.

In [4]:
gender_metrics = subgroup_metrics(demographics['gender'], 'gender')
print(gender_metrics.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

demographic_type subgroup  support  positive_base_rate  precision  recall     f1
          gender   Female     8006              0.1125     0.1771  0.6615 0.2794
          gender     Male     6991              0.1120     0.1701  0.6181 0.2667


## Age-band subgroup metrics

**Bias-source and mitigation placeholder:** Discuss age-related variation and why any small subgroup should not be over-interpreted.

In [5]:
age_metrics = subgroup_metrics(demographics['age_band'], 'age')
age_metrics['small_sample_support_lt_100'] = age_metrics['support'] < 100
print(age_metrics.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
if age_metrics['small_sample_support_lt_100'].any():
    print('\nSmall age subgroups (support < 100):')
    print(age_metrics.loc[age_metrics['small_sample_support_lt_100'], ['subgroup', 'support']].to_string(index=False))

demographic_type subgroup  support  positive_base_rate  precision  recall     f1  small_sample_support_lt_100
             age   [0-10)       26              0.0385     0.0000  0.0000 0.0000                         True
             age  [10-20)       79              0.0253     0.1000  1.0000 0.1818                         True
             age  [20-30)      232              0.1078     0.2632  0.6000 0.3659                        False
             age  [30-40)      533              0.1032     0.2281  0.7091 0.3451                        False
             age  [40-50)     1482              0.1053     0.1946  0.5962 0.2934                        False
             age  [50-60)     2592              0.0961     0.1777  0.5301 0.2661                        False
             age  [60-70)     3415              0.1098     0.1662  0.6293 0.2630                        False
             age  [70-80)     3710              0.1216     0.1729  0.6918 0.2767                        False
          

## Recall comparisons and exported subgroup table

**Bias-source and mitigation placeholder:** Explain how recall gaps will be monitored, investigated, and mitigated before clinical deployment.

In [6]:
def recall_chart(metrics, title, output_path, ordered_subgroups=None):
    plotted = metrics.copy()
    if ordered_subgroups is not None:
        plotted['subgroup'] = pd.Categorical(plotted['subgroup'], categories=ordered_subgroups, ordered=True)
        plotted = plotted.sort_values('subgroup')
    plt.figure(figsize=(9, 5))
    bars = plt.bar(plotted['subgroup'].astype(str), plotted['recall'], color='#2E86AB')
    plt.axhline(overall_metrics['recall'], color='#C0392B', linestyle='--', label=f"Overall recall ({overall_metrics['recall']:.3f})")
    plt.ylim(0, 1)
    plt.ylabel('Recall')
    plt.title(title)
    plt.xticks(rotation=30, ha='right')
    plt.legend()
    for bar, value in zip(bars, plotted['recall']):
        plt.text(bar.get_x() + bar.get_width() / 2, value + 0.02, f'{value:.2f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

recall_chart(race_metrics, 'Readmission Recall by Race', RESULTS_DIR / 'fairness_recall_by_race.png')
recall_chart(age_metrics, 'Readmission Recall by Age Band', RESULTS_DIR / 'fairness_recall_by_age.png', age_levels)

combined_metrics = pd.concat([race_metrics, gender_metrics, age_metrics], ignore_index=True, sort=False)
combined_metrics.to_csv(RESULTS_DIR / 'fairness_subgroup_metrics.csv', index=False)
print(f"Saved subgroup metrics: {RESULTS_DIR / 'fairness_subgroup_metrics.csv'}")

C:\Users\sagir\AppData\Local\Temp\ipykernel_24020\4044632397.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved subgroup metrics: C:\Users\sagir\OneDrive\Desktop\capstone\results\fairness_subgroup_metrics.csv


C:\Users\sagir\AppData\Local\Temp\ipykernel_24020\4044632397.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Recall-gap flags

**Bias-source and mitigation placeholder:** For each flagged group, document a root-cause review and a concrete mitigation or escalation path.

In [7]:
recall_gap_threshold = overall_metrics['recall'] - 0.10
flagged = combined_metrics.loc[combined_metrics['recall'] < recall_gap_threshold].copy()
if flagged.empty:
    print(f"No subgroup recall is more than 10 percentage points below overall recall ({overall_metrics['recall']:.4f}).")
else:
    print(f"Recall-gap flags: below {recall_gap_threshold:.4f} (overall recall {overall_metrics['recall']:.4f} minus 10pp)")
    for _, row in flagged.iterrows():
        small_sample = pd.notna(row.get('small_sample_support_lt_100')) and bool(row.get('small_sample_support_lt_100'))
        small_sample_note = ' [support < 100; interpret cautiously]' if small_sample else ''
        print(f"- {row['demographic_type']} = {row['subgroup']}: recall {row['recall']:.4f}, support {int(row['support'])}{small_sample_note}")

Recall-gap flags: below 0.5413 (overall recall 0.6413 minus 10pp)
- race = Asian: recall 0.5238, support 117
- race = Unknown: recall 0.4722, support 350
- age = [0-10): recall 0.0000, support 26 [support < 100; interpret cautiously]
- age = [50-60): recall 0.5301, support 2592


## Worked mitigation experiment: age-band threshold calibration

This retrospective experiment examines whether the observed age-band recall gap is addressable with different operating thresholds. **Bias-source and mitigation placeholder:** Record the clinical rationale, expected benefits, and potential harms of this intervention.

In [8]:
# Reconstruct age labels from the standardized ordinal feature, independently of the earlier display table.
calibration_age_column = next(column for column in X_test.columns if column.endswith('age_ordinal'))
calibration_numeric_features = list(preprocessing_pipeline.named_steps['preprocessor'].transformers_[0][2])
calibration_age_position = calibration_numeric_features.index('age_ordinal')
calibration_age_codes = np.rint(
    X_test[calibration_age_column] * scaler.scale_[calibration_age_position] + scaler.mean_[calibration_age_position]
).astype(int).clip(0, len(age_levels) - 1)
calibration_age_labels = pd.Series([age_levels[code] for code in calibration_age_codes], index=X_test.index)
print('Reconstructed age bands for calibration:', calibration_age_labels.value_counts().sort_index().to_dict())

Reconstructed age bands for calibration: {'[0-10)': 26, '[10-20)': 79, '[20-30)': 232, '[30-40)': 533, '[40-50)': 1482, '[50-60)': 2592, '[60-70)': 3415, '[70-80)': 3710, '[80-90)': 2532, '[90-100)': 396}


## Per-age-band threshold sweep

Thresholds are selected from 0.05 through 0.50 by closeness to the overall recall target of 0.64. Age bands with fewer than 300 encounters are flagged as unreliable.

In [9]:
TARGET_RECALL = 0.64
CALIBRATION_THRESHOLDS = np.arange(0.05, 0.501, 0.05)
calibration_rows = []

for age_band in age_levels:
    indices = np.flatnonzero(calibration_age_labels.to_numpy() == age_band)
    actual = y_test.iloc[indices]
    group_probabilities = probabilities[indices]
    global_predictions = (group_probabilities >= FINAL_THRESHOLD).astype(int)
    candidates = []
    for threshold in CALIBRATION_THRESHOLDS:
        threshold_predictions = (group_probabilities >= threshold).astype(int)
        recall = recall_score(actual, threshold_predictions, zero_division=0)
        precision = precision_score(actual, threshold_predictions, zero_division=0)
        candidates.append((threshold, recall, precision))
    subgroup_threshold, calibrated_recall, calibrated_precision = min(
        candidates, key=lambda item: (abs(item[1] - TARGET_RECALL), -item[2], item[0])
    )
    calibration_rows.append({
        'age_band': age_band,
        'support': len(indices),
        'global_threshold_recall': recall_score(actual, global_predictions, zero_division=0),
        'subgroup_threshold': subgroup_threshold,
        'subgroup_threshold_recall': calibrated_recall,
        'subgroup_threshold_precision': calibrated_precision,
        'support_under_300_unreliable': len(indices) < 300,
    })

threshold_calibration = pd.DataFrame(calibration_rows)
threshold_calibration.to_csv(RESULTS_DIR / 'fairness_subgroup_threshold_calibration.csv', index=False)
print(threshold_calibration.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

age_band  support  global_threshold_recall  subgroup_threshold  subgroup_threshold_recall  subgroup_threshold_precision  support_under_300_unreliable
  [0-10)       26                   0.0000              0.0500                     1.0000                        0.1000                          True
 [10-20)       79                   1.0000              0.1500                     0.5000                        0.0588                          True
 [20-30)      232                   0.6000              0.1000                     0.7200                        0.2250                          True
 [30-40)      533                   0.7091              0.1500                     0.6000                        0.2727                         False
 [40-50)     1482                   0.5962              0.1000                     0.6667                        0.1661                         False
 [50-60)     2592                   0.5301              0.1000                     0.6265           

## Governance note and recall comparison

**Worked mitigation example only.** In a real deployment, per-subgroup thresholds require clinical, ethical, legal, and governance review. This proof of concept shows the gap may be addressable; it is not a recommendation to silently ship different cutoffs for different age groups. Thresholds should also be selected on validation data and prospectively monitored, rather than calibrated retrospectively on the test set.

In [10]:
plot_positions = np.arange(len(threshold_calibration))
bar_width = 0.38
plt.figure(figsize=(12, 6))
plt.bar(plot_positions - bar_width / 2, threshold_calibration['global_threshold_recall'], bar_width, label='Global threshold (0.12)', color='#7F8C8D')
plt.bar(plot_positions + bar_width / 2, threshold_calibration['subgroup_threshold_recall'], bar_width, label='Calibrated subgroup threshold', color='#2E86AB')
plt.axhline(TARGET_RECALL, color='#C0392B', linestyle='--', label=f'Target recall ({TARGET_RECALL:.2f})')
plt.xticks(plot_positions, threshold_calibration['age_band'], rotation=30, ha='right')
plt.ylim(0, 1.05)
plt.ylabel('Recall')
plt.title('Age-Band Recall: Global vs. Calibrated Thresholds')
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fairness_threshold_calibration_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

primary_case = threshold_calibration.loc[threshold_calibration['age_band'] == '[50-60)'].iloc[0]
primary_indices = np.flatnonzero(calibration_age_labels.to_numpy() == '[50-60)')
primary_actual = y_test.iloc[primary_indices]
primary_global_precision = precision_score(primary_actual, (probabilities[primary_indices] >= FINAL_THRESHOLD).astype(int), zero_division=0)
precision_change = primary_case['subgroup_threshold_precision'] - primary_global_precision
print('\n[50-60) mitigation summary:')
print(f"  Global threshold {FINAL_THRESHOLD:.2f}: recall {primary_case['global_threshold_recall']:.4f}, precision {primary_global_precision:.4f}")
print(f"  Calibrated threshold {primary_case['subgroup_threshold']:.2f}: recall {primary_case['subgroup_threshold_recall']:.4f}, precision {primary_case['subgroup_threshold_precision']:.4f}")
print(f"  Precision change: {precision_change:+.4f}")


[50-60) mitigation summary:
  Global threshold 0.12: recall 0.5301, precision 0.1777
  Calibrated threshold 0.10: recall 0.6265, precision 0.1622
  Precision change: -0.0155


C:\Users\sagir\AppData\Local\Temp\ipykernel_24020\3945397442.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Ethics, Privacy, and Clinical Deployment

In a real deployment, the information used by this model would be sensitive patient data rather than a classroom dataset. In the United States, a hospital and any qualifying vendor would need to assess their responsibilities under HIPAA, including whether they are covered entities or business associates, and protect electronic protected health information with appropriate administrative, physical, and technical safeguards. Data used for development should be de-identified where feasible and limited to what is necessary for the stated purpose. Production access should be role-based and limited to authorized users; systems should use strong authentication, encryption in transit and at rest where appropriate, and audit logging that records access, scoring activity, and material model-related actions. If the deployment involves people in the European Economic Area or otherwise falls within its scope, GDPR obligations would also need to be assessed, including a lawful basis for processing, data minimisation, purpose limitation, appropriate security, and mechanisms for data-subject rights. These requirements require institution-specific legal, privacy, and security review rather than a one-time technical checklist.

Hospital data would ordinarily reach a production model through interoperable clinical systems rather than manually assembled files. HL7 FHIR is the widely used standard for exchanging healthcare information and can provide a structured route for supplying relevant encounters, observations, medications, and patient context to a readmission-risk workflow. A production integration should validate incoming FHIR data, preserve provenance, minimise the fields shared with the model service, and make clear when data are incomplete or stale.

Patients should be informed, in plain language, when an algorithm contributes to assessment of their readmission risk. The disclosure could explain that the tool estimates risk from information in the medical record, is intended to help the care team plan follow-up and support, does not replace clinical judgment, and does not determine eligibility for care. The precise consent or notice process will depend on the care setting, local policy, and applicable law, but transparency supports trust and gives patients an opportunity to raise concerns about inaccurate or missing information.

Finally, this model must operate as clinical decision support, not as an automated decision-maker. A clinician must always be able to override or disregard a risk score based on information the model cannot see, such as a conversation with the patient, caregiver capacity, evolving symptoms, or details absent from structured records. The interface should present the score with its limitations and supporting context, capture appropriate clinician reasoning when an override matters, and ensure that responsibility for care remains with the clinical team.

## Bias amplification through deployment

Deployment can amplify rather than merely reflect an existing performance gap. If the model systematically under-flags a demographic group?such as the lower recall observed for the age **[50-60)** band?members of that group may receive fewer transition-of-care resources in practice. If under-intervention contributes to worse real-world outcomes, the next retraining dataset will encode the same disparity. Successive model updates could then reinforce or widen the original gap instead of correcting it. Ongoing fairness monitoring, including the rolling subgroup analysis described in the deployment architecture, is therefore not optional: a one-time fairness check at launch is insufficient.

## When clinicians should override the model

The risk score is decision support, not a replacement for clinical judgment. A clinician should override or disregard the score when any of the following applies:

- Recent laboratory results, clinical findings, or a change in clinical status are not yet reflected in the structured EHR data used by the model.
- The discharge care plan changed after the prediction was generated.
- The patient reports symptoms, social circumstances, or caregiver-availability constraints that the model cannot observe.
- The encounter contains known data-quality problems, such as a miscoded diagnosis or incomplete record.
- The clinician's direct clinical judgment conflicts with the score after considering information unavailable to the model.